In [1]:
import socket
import asyncio
from fastmcp import FastMCP, Client

In [2]:
import os
def make_dir():
    if os.path.exists("path"):
        print("Path directory already exists")
    else:
        print("✗ Path directory doesn't exist - creating it...")
        os.makedirs("path")
        print("✓ Path directory created")

In [3]:
PORT = 8000
def test_port(port = PORT):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(('127.0.0.1',PORT))
            return False
        except socket.error:
            return True
f"Port {PORT} is available: {not test_port()}"            

'Port 8000 is available: True'

In [4]:
def print_stream_info(read, write, _sid, verbose = False):
    """Print information about the read/write streams and session ID.
    """
    if verbose:
        print("READ (receives FROM server):")
        print(read)
        print()
        
        print("WRITE (sends TO server):")
        print(write)
        print()
        
        print("SESSION ID:")
        print(_sid())

In [5]:
from langchain_core.tools import tool

In [6]:
@tool
def multiply(a:int , b:int)-> int:
    """Multiply 2 numbers """
    return a*b
print(multiply.name)
print(multiply.description)
print(multiply.args)
print("What is 2 x 3?")
print("Answer: " + str(multiply.invoke({"a": 2, "b": 3})))

multiply
Multiply 2 numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
What is 2 x 3?
Answer: 6


In [7]:
from fastmcp import FastMCP
mcp = FastMCP(
    name = "Calculator mcp server",
    instructions = """
        This server provides data analysis tools.
        Call get_average() to analyze numerical data.
    """)
print("mcp ",mcp)


mcp  FastMCP('Calculator mcp server')


In [8]:
@mcp.tool
def add(a:int ,b:int) -> int:
    """
    Adds 2 integers together
    Args:
    a(int)- The first integer to be added
    b(int)- The second integer to be added

    Returns:
    int: sum of a and b
    Example:
    add(3,5)
    8
    """
    return a + b

@mcp.tool
def subtract(a: int, b: int) -> int:
    """
    Subtract one integer from another.

    Args:
        a (int): The number to subtract from.
        b (int): The number to subtract.

    Returns:
        int: The result of `a - b`.

    Example:
        >>> subtract(10, 4)
        6
    """
    return a - b


In [9]:
@mcp.resource("file:///endpoint/{name}")
def return_template_document(name:str) -> str:
    """ Read a document by name """
    return f"Document contents of {name}"

In [10]:
make_dir()

Path directory already exists


In [15]:
!wget -P path/ https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/aNE__JjH4DLNEibuNpfDlg/examples.txt
!wget -P path/ https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/tfoeGPInNoajVS0DSohdVg/README.txt

--2026-02-18 09:47:24--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/aNE__JjH4DLNEibuNpfDlg/examples.txt
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 198.23.119.245
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|198.23.119.245|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 0 [text/plain]
Saving to: ‘path/examples.txt’

examples.txt            [ <=>                ]       0  --.-KB/s    in 0s      

2026-02-18 09:47:25 (0.00 B/s) - ‘path/examples.txt’ saved [0/0]

--2026-02-18 09:47:25--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/tfoeGPInNoajVS0DSohdVg/README.txt
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 198.23.119.245
Connecting to cf-courses-data.s3.

In [11]:
@mcp.resource("file://endpoint2/{name}")
def read_document(name:str) -> str:
    """Reads document name by path directory"""
    try:
        #Read from the actual file path system
        with open(f"path/{name}", "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"Document '{name}' not found in path directory"
    except Exception as e:
        return f"Error reading document: {str(e)}"        

In [12]:
@mcp.prompt(title = "Code Review")
def review_code(code: str) -> str:
    return f"Please review this code:\n\n{code}"


In [13]:
from fastmcp import Client
client = Client(mcp)
print(f"Client:{client}")                

Client:<fastmcp.client.client.Client object at 0x1087921d0>


In [14]:
async def call_add_tool(a:int , b:int):
    async with client:
        result = await client.call_tool("add",{"a": a, "b": b})
        return  result 

In [15]:
result = await call_add_tool(5,6)
result

CallToolResult(content=[TextContent(type='text', text='11', annotations=None, meta=None)], structured_content={'result': 11}, meta=None, data=11, is_error=False)

In [16]:
# The actual answer/data
print("\nResult Data .data :")
print(result.data)  # 9

# Content (text format)
print("\nContent (as text):")
print(result.content[0].text)  # "9"

# Structured content (as dictionary)
print("\nStructured Content:")
print(result.structured_content)  


Result Data .data :
11

Content (as text):
11

Structured Content:
{'result': 11}


In [17]:
async def  call_subtract_tool(a:int , b:int):
    async with client:
        result = await client.call_tool("subtract",{"a":a, "b": b})
        return result

In [18]:
result = await call_subtract_tool(6,5)
result

CallToolResult(content=[TextContent(type='text', text='1', annotations=None, meta=None)], structured_content={'result': 1}, meta=None, data=1, is_error=False)

In [19]:
#Listing all the tools
async with client:
    tools = await client.list_tools()
    for tool in tools:
        print(f"{tool.name}:{tool.description}")

add:Adds 2 integers together
Args:
a(int)- The first integer to be added
b(int)- The second integer to be added

Returns:
int: sum of a and b
Example:
add(3,5)
8
subtract:Subtract one integer from another.

Args:
    a (int): The number to subtract from.
    b (int): The number to subtract.

Returns:
    int: The result of `a - b`.

Example:
    >>> subtract(10, 4)
    6


In [20]:
tool_obj = tools[0]
print(tool_obj)

name='add' title=None description='Adds 2 integers together\nArgs:\na(int)- The first integer to be added\nb(int)- The second integer to be added\n\nReturns:\nint: sum of a and b\nExample:\nadd(3,5)\n8' inputSchema={'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'} outputSchema={'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True} icons=None annotations=None meta={'_fastmcp': {'tags': []}} execution=None


In [21]:
input_schema = tool.inputSchema
print(input_schema)

{'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}


In [22]:
output_schema = tool.outputSchema
print(output_schema)

{'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}


In [23]:
async def call_resource(name):
    async with client:
        result = await client.read_resource(f"file:///endpoint/{name}")
        return result

In [24]:
response = await call_resource("README.txt")
print(response[0].text)

Document contents of README.txt


In [28]:
async def call_resource2(name):
    async with client:
        result = await client.read_resource(f"file://endpoint2/{name}")
        return result

In [29]:
response = await call_resource2("README.txt")


In [32]:
response = await call_resource2("random.txt")
resource = response[0]
resource

TextResourceContents(uri=AnyUrl('file://endpoint2/random.txt'), mimeType='text/plain', meta=None, text="Document 'random.txt' not found in path directory")

In [33]:
print(f"uri:      {resource.uri}")
print(f"mimeType: {resource.mimeType}")
print(f"meta:     {resource.meta}")
print(f"text:     {resource.text}")

uri:      file://endpoint2/random.txt
mimeType: text/plain
meta:     None
text:     Document 'random.txt' not found in path directory


In [34]:
async def call_prompt(code):
    async with client:

        result = await client.get_prompt("review_code", {"code": code})
        return result

In [35]:
response = await call_prompt("CODE TO BE REVIEWED")


In [36]:
message=response.messages[0]
print(f"Prompt Role:{message.role}")
print(f"Prompt Content:{message.content.text}")

Prompt Role:user
Prompt Content:Please review this code:

CODE TO BE REVIEWED


In [37]:
f"Port {PORT} is available: {not test_port()}"

'Port 8000 is available: True'

In [38]:
asyncio.create_task(mcp.run_http_async(port = PORT))
print(f"HTTP MCP Server started in background on port {PORT}")

HTTP MCP Server started in background on port 8000




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 2.14.5                                │
│                            https://gofastmcp.com                             │
│                                                                              │
│                    🖥  Server:      Calculator mcp server                     │
│                    🚀 Deploy free: https://fastmcp.cloud                     │
│                          

[02/18/26 10:19:17] INFO     Starting MCP server 'Calculator mcp server' with transport 'http' on    ]8;id=81801;file:///Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.11/site-packages/fastmcp/server/server.py\server.py]8;;\:]8;id=307065;file:///Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.11/site-packages/fastmcp/server/server.py#2580\2580]8;;\
                             http://127.0.0.1:8000/mcp                                                             

INFO:     Started server process [83759]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [39]:
from fastmcp.client.transports import StdioTransport ,StreamableHttpTransport
transport_http = StreamableHttpTransport(
    url = f"http://127.0.0.1:{PORT}/mcp"
)

In [40]:
http_client = Client(transport_http)
print('http_client',http_client )


http_client <fastmcp.client.client.Client object at 0x10e0ad5d0>


In [41]:
async def test_client_http(client: Client, a: int, b: int)->int:
    async with client:  
        result = await client.call_tool("add", {"a": a, "b": b})
        return result

In [42]:
response = await test_client_http(http_client, 4, 5)
print(response.content[0].text)

INFO:     127.0.0.1:49355 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49356 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:49357 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49358 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49359 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49360 - "DELETE /mcp HTTP/1.1" 200 OK
9


In [43]:
async def get_tools (client: Client):
    async with client:
        tools = await client.list_tools()
        return tools

In [44]:
tools = await get_tools(http_client)
tools

INFO:     127.0.0.1:49419 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49420 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:49421 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49422 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49423 - "DELETE /mcp HTTP/1.1" 200 OK


[Tool(name='add', title=None, description='Adds 2 integers together\nArgs:\na(int)- The first integer to be added\nb(int)- The second integer to be added\n\nReturns:\nint: sum of a and b\nExample:\nadd(3,5)\n8', inputSchema={'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, outputSchema={'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}, icons=None, annotations=None, meta={'_fastmcp': {'tags': []}}, execution=None),
 Tool(name='subtract', title=None, description='Subtract one integer from another.\n\nArgs:\n    a (int): The number to subtract from.\n    b (int): The number to subtract.\n\nReturns:\n    int: The result of `a - b`.\n\nExample:\n    >>> subtract(10, 4)\n    6', inputSchema={'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, outputSchema={'properties': {'result': {'type': 'integer'}}, 

In [55]:
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from mcp import ClientSession
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature = 0,
    groq_api_key = os.environ["GROQ_API_KEY"]
)

In [48]:
from mcp.client.streamable_http import streamable_http_client
async with streamable_http_client(f"http://127.0.0.1:{PORT}/mcp") as (read, write , _sid):
    print_stream_info(read, write , _sid ,verbose = True)

READ (receives FROM server):
MemoryObjectReceiveStream(_state=_MemoryObjectStreamState(max_buffer_size=0, buffer=deque([]), open_send_channels=1, open_receive_channels=1, waiting_receivers=OrderedDict(), waiting_senders=OrderedDict()), _closed=False)

WRITE (sends TO server):
MemoryObjectSendStream(_state=_MemoryObjectStreamState(max_buffer_size=0, buffer=deque([]), open_send_channels=1, open_receive_channels=1, waiting_receivers=OrderedDict(), waiting_senders=OrderedDict()), _closed=False)

SESSION ID:
None


In [56]:
async with streamable_http_client(f"http://127.0.0.1:{PORT}/mcp") as (read, write, _sid):
    async with ClientSession(read,write) as session:
        await session.initialize()
        #Loading tools from live mcp session
        tools = await load_mcp_tools(session)
        #Build the agent while session is still open
        agent = create_react_agent(
            model = llm,
            tools = tools
        )
        agent_response = await agent.ainvoke({"messages": "Use the add tool to add 2 and 1 and let me know if you used a tool."})

INFO:     127.0.0.1:54343 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54344 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:54345 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54346 - "POST /mcp HTTP/1.1" 200 OK


/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_83759/342862452.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


INFO:     127.0.0.1:54348 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54349 - "DELETE /mcp HTTP/1.1" 200 OK


In [53]:
import os,getpass
def _set_if_undefined(var:str):
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)
_set_if_undefined("GROQ_API_KEY")    

GROQ_API_KEY ········


In [57]:
print(agent_response['messages'][-1].content)

Yes, I used the add tool to calculate the sum of 2 and 1.


In [58]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

In [59]:
client = MultiServerMCPClient(
    {
        "stdio-client": {
            "command": "python",
            "args": ["stdio_server.py"],
            "transport": "stdio"
        },
        "http-client": {
            "url": f"http://127.0.0.1:{PORT}/mcp",
            "transport": "streamable_http"
        }
    }
)

In [60]:
tools = await client.get_tools()
[tool.name for tool in tools]

INFO:     127.0.0.1:54435 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54436 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:54437 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54438 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54439 - "DELETE /mcp HTTP/1.1" 200 OK


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/contextlib.py:105: DeprecationWarning: Use `streamable_http_client` instead.
  self.gen = func(*args, **kwds)
  + Exception Group Traceback (most recent call last):
  |   File "/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_83759/3093703347.py", line 1, in <module>
  |     tools = await client.get_tools()
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.11/site-packages/langchain_mcp_adapters/client.py", line 197, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/l